# OSML Jupyter Extension - Publishing Layers Example

This notebook demonstrates how to programmatically create and publish overlay layers to the OSML image viewer from within a Jupyter notebook. This approach allows you to integrate your analysis workflows directly with the visualization.

## Prerequisites

1. **Open an image** using the OSML Jupyter Extension context menu (right-click → "OversightML: Open")
2. **Select the same kernel** for this notebook as the one used by the image viewer (typically `osml-kernel`)
3. **Update the image_name** variable below to match your opened image file

## Important Notes About Feature Coordinates

When creating features for overlay, you can specify locations in **pixel coordinates** using either:

- **`imageBBox`**: For simple rectangular regions `[min_x, min_y, max_x, max_y]`
- **`imageGeometry`**: For complex shapes (Point, LineString, Polygon) with coordinates as `[x, y]` pixel positions

The coordinate system uses the **top-left corner as (0,0)** with x increasing rightward and y increasing downward.

In [6]:
# Import required libraries
import geojson

# IMPORTANT: Update this to match your opened image file. This must match the full path
# of the image as shown in the Jupyter file browser
image_name = "demo-folder/large.tif"  # Replace with your actual image filename

In [7]:
def publish_overlay(image_name, collection_name, fc):
    accessor = ImagedFeaturePropertyAccessor()
    for f in fc['features']:
        geom = accessor.find_image_geometry(f)
        accessor.set_image_geometry(f, geom)
            
    tile_index = STRFeature2DSpatialIndex(fc, use_image_geometries=True)
    key = f"{image_name}:{collection_name}"
    global_cache_manager.set_overlay_factory(key, tile_index)    

## Example 1: Object Detection Results

Create point features with bounding boxes using `imageBBox` property. This is ideal for object detection results where you have rectangular regions of interest.

In [8]:
def create_bbox_results():
    """Create sample object detection results as point features with bounding boxes."""
    detection_features = []
    
    # Simulate detection results across the image
    detections = [
        {"center": (1024, 768), "size": 50, "confidence": 0.95, "class": "vehicle"},
        {"center": (2048, 1536), "size": 75, "confidence": 0.87, "class": "building"},
        {"center": (3072, 2304), "size": 30, "confidence": 0.72, "class": "vehicle"},
        {"center": (1500, 1200), "size": 40, "confidence": 0.83, "class": "aircraft"},
    ]
    
    for i, detection in enumerate(detections):
        x, y = detection["center"]
        half_size = detection["size"] // 2
        
        # Create feature with imageBBox (pixel coordinates)
        feature = geojson.Feature(
            geometry=None,  # No geographic coordinates yet - image annotation
            properties={
                "imageBBox": [x - half_size, y - half_size, x + half_size, y + half_size],
                "confidence": detection["confidence"],
                "object_class": detection["class"],
                "detection_id": i + 1
            }
        )
        detection_features.append(feature)
    
    return geojson.FeatureCollection(features=detection_features)

# Create and publish detection results
detections = create_bbox_results()
publish_overlay(image_name, "example_bbox", detections)
print(f"Published {len(detections['features'])} object detections")

Published 4 object detections


## Example 2: Road Network with LineString Geometry

Create linear features using `imageGeometry` with `LineString` type. This is perfect for roads, rivers, or any linear features.

In [9]:
def create_road_network():
    """Create sample road network using LineString geometries."""
    road_features = []
    
    # Define road segments as pixel coordinates [x, y]
    roads = [
        {
            "name": "Main Street",
            "coordinates": [[100, 200], [500, 200], [500, 800], [900, 800]],
            "road_type": "primary"
        },
        {
            "name": "Oak Avenue", 
            "coordinates": [[200, 100], [200, 600], [700, 600]],
            "road_type": "secondary"
        },
        {
            "name": "River Road",
            "coordinates": [[50, 500], [150, 450], [250, 400], [350, 380], [450, 350]],
            "road_type": "tertiary"
        }
    ]
    
    for road in roads:
        feature = geojson.Feature(
            geometry=None,  # No geographic coordinates yet - will be calculated by OSML
            properties={
                "imageGeometry": {
                    "type": "LineString",
                    "coordinates": road["coordinates"]  # Pixel coordinates (x, y)
                },
                "name": road["name"],
                "road_type": road["road_type"],
            }
        )
        road_features.append(feature)
    
    return geojson.FeatureCollection(features=road_features)

# Create and publish road network
roads = create_road_network()
publish_overlay(image_name, "example_road_network", roads)
print(f"Published {len(roads['features'])} road segments")

Published 3 road segments


## Example 3: Regions with Polygon Geometry

Create area features using `imageGeometry` with `Polygon` type. This is ideal for more complex features or segmentation results.

In [10]:
def create_regions():
    """Create sample analysis regions using Polygon geometries."""
    region_features = []
    
    # Define regions of interest as polygons
    # Note: First and last coordinates must be the same to close the polygon
    regions = [
        {
            "name": "Urban Area",
            "coordinates": [[[500, 500], [1500, 500], [1500, 1200], [500, 1200], [500, 500]]],
            "land_use": "urban"
        },
        {
            "name": "Forest Region",
            "coordinates": [[[2000, 1000], [3000, 1000], [2800, 2000], [2200, 2000], [2000, 1000]]],
            "land_use": "forest"
        },
        {
            "name": "Agricultural Area",
            "coordinates": [[[1000, 2500], [2000, 2500], [2000, 3500], [1000, 3500], [1000, 2500]]],
            "land_use": "agriculture"
        }
    ]
    
    for region in regions:
        feature = geojson.Feature(
            geometry=None,  # No geographic coordinates yet - will be calculated by OSML
            properties={
                "imageGeometry": {
                    "type": "Polygon",
                    "coordinates": region["coordinates"]  # Pixel coordinates (x, y)
                },
                "name": region["name"],
                "land_use": region["land_use"]
            }
        )
        region_features.append(feature)
    
    return geojson.FeatureCollection(features=region_features)

# Create and publish regions
regions = create_regions()
publish_overlay(image_name, "example_regions", regions)
print(f"Published {len(regions['features'])} regions")

Published 3 analysis regions


## Key Takeaways

- Use **`imageBBox`** for simple rectangular regions: `[min_x, min_y, max_x, max_y]`
- Use **`imageGeometry`** for complex shapes with coordinate arrays: `[[x, y], [x, y], ...]`
- Pixel coordinates start at **top-left (0,0)** with x→ and y↓
- OSML automatically converts pixel coordinates to geographic coordinates
- Layer names should be descriptive and unique per image
- You can update layers dynamically as your analysis evolves